In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/features/train_numeric_features.csv")

X = train_df.drop(columns=["demand"])
y = train_df["demand"]

print(X.shape)

(73459, 13)


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [4]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
).reshape(-1, 1)

y_val = torch.tensor(
    y_val.values,
    dtype=torch.float32
).reshape(-1, 1)

In [5]:
import torch.nn as nn

class DemandNN(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

model = DemandNN(X_train.shape[1])

print(model)

DemandNN(
  (network): Sequential(
    (0): Linear(in_features=13, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): ReLU()
    (10): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [6]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [7]:
from sklearn.metrics import r2_score
import numpy as np

epochs = 200

best_val_rmse = np.inf
patience = 20
counter = 0

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    pred = model(X_train)

    loss = criterion(pred, y_train)

    loss.backward()

    optimizer.step()

    # validation
    model.eval()

    with torch.no_grad():

        val_pred = model(X_val)

        val_rmse = torch.sqrt(
            criterion(val_pred, y_val)
        ).item()

    if val_rmse < best_val_rmse:

        best_val_rmse = val_rmse

        torch.save(
            model.state_dict(),
            "best_nn_model.pth"
        )

        counter = 0

    else:
        counter += 1

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:3d} | "
            f"Train Loss {loss.item():.6f} | "
            f"Val RMSE {val_rmse:.6f}"
        )

    if counter >= patience:

        print("Early stopping")
        break

Epoch   0 | Train Loss 0.115795 | Val RMSE 0.203462
Epoch  10 | Train Loss 0.020483 | Val RMSE 0.125727
Epoch  20 | Train Loss 0.012644 | Val RMSE 0.092164
Epoch  30 | Train Loss 0.009647 | Val RMSE 0.076867
Epoch  40 | Train Loss 0.008647 | Val RMSE 0.075447
Epoch  50 | Train Loss 0.007711 | Val RMSE 0.074639
Epoch  60 | Train Loss 0.007257 | Val RMSE 0.074099
Epoch  70 | Train Loss 0.007024 | Val RMSE 0.073256
Epoch  80 | Train Loss 0.006793 | Val RMSE 0.072675
Epoch  90 | Train Loss 0.006535 | Val RMSE 0.072121
Epoch 100 | Train Loss 0.006322 | Val RMSE 0.071870
Epoch 110 | Train Loss 0.006146 | Val RMSE 0.071489
Epoch 120 | Train Loss 0.006036 | Val RMSE 0.071072
Epoch 130 | Train Loss 0.005914 | Val RMSE 0.070821
Epoch 140 | Train Loss 0.005732 | Val RMSE 0.070680
Epoch 150 | Train Loss 0.005674 | Val RMSE 0.070421
Epoch 160 | Train Loss 0.005576 | Val RMSE 0.070268
Epoch 170 | Train Loss 0.005487 | Val RMSE 0.070106
Epoch 180 | Train Loss 0.005414 | Val RMSE 0.069941
Epoch 190 | 

In [8]:
model.load_state_dict(
    torch.load("best_nn_model.pth")
)

model.eval()

with torch.no_grad():

    preds = model(X_val)

preds = preds.numpy().flatten()

rmse = np.sqrt(
    np.mean(
        (preds - y_val.numpy().flatten())**2
    )
)

r2 = r2_score(
    y_val.numpy().flatten(),
    preds
)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 0.06980329
R2: 0.7675056457519531
